### Configuración
Misma celda de configuración que la Fase 3: rutas, importaciones y funciones auxiliares.

In [2]:
# ---------------------------------------------------------------------------
# Configuración, rutas e importaciones
# ---------------------------------------------------------------------------
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Respaldo de visualización: el notebook funciona con o sin IPython instalado.
try:
    from IPython.display import SVG, display
    _TIENE_IPYTHON = True
except ImportError:  # pragma: no cover
    _TIENE_IPYTHON = False

    def display(*args, **kwargs):
        pass

    class SVG:  # marcador de posición para poder usar display(SVG(...))
        def __init__(self, datos):
            self.datos = datos

RAIZ = Path.cwd()
if not (RAIZ / "src").exists() and (RAIZ.parent / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import graficos_svg as g   # librería de gráficos SVG del proyecto

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RUTA_DATOS = RAIZ / "datos"
RUTA_REPORTES = RAIZ / "reports"
RUTA_GRAFICOS = RAIZ / "graficos"
RUTA_GRAFICOS.mkdir(exist_ok=True)
DIR_ENTREGA = RAIZ / "entrega"
DIR_ENTREGA.mkdir(exist_ok=True)


print(f"IPython disponible: {_TIENE_IPYTHON}")
print(f"pandas {pd.__version__} · numpy {np.__version__}")


def mostrar(svg: str, nombre: str | None = None) -> None:
    """Renderiza un SVG en el notebook y, si se indica, lo exporta a graficos/."""
    if nombre:
        (RUTA_GRAFICOS / f"{nombre}.svg").write_text(svg, encoding="utf-8")
    if _TIENE_IPYTHON:
        display(SVG(svg))
    else:
        print(f"[gráfico {nombre or 'sin nombre'}: {len(svg):,} caracteres SVG]")


def tabla_md(df: pd.DataFrame, dec: int = 2) -> str:
    """Convierte un DataFrame en tabla Markdown (sin depender de tabulate)."""
    enc = [str(c) for c in df.columns]
    filas = []
    for _, r in df.iterrows():
        vals = []
        for v in r:
            if isinstance(v, (int, np.integer)):
                vals.append(f"{v:,}")
            elif isinstance(v, (float, np.floating)):
                vals.append(f"{v:,.{dec}f}")
            else:
                vals.append(str(v))
        filas.append(vals)
    lineas = ["| " + " | ".join(enc) + " |",
              "|" + "|".join(["---"] * len(enc)) + "|"]
    lineas += ["| " + " | ".join(f) + " |" for f in filas]
    return "\n".join(lineas)

IPython disponible: True
pandas 3.0.6 · numpy 2.5.3


### Carga de datos de la Fase 3

In [3]:
# Tablas limpias exportadas al final de la Fase 3 (celda 7.3)
df = pd.read_csv(RUTA_DATOS / "analitica_ventas_linea_notebook.csv",
                 encoding="utf-8-sig", parse_dates=["Order Date", "Ship Date"])
pedidos = pd.read_csv(RUTA_DATOS / "analitica_pedidos_notebook.csv",
                      encoding="utf-8-sig", parse_dates=["Fecha"])
print(f"Grano línea : {df.shape[0]:,} × {df.shape[1]}   |   Grano pedido: {pedidos.shape[0]:,} × {pedidos.shape[1]}")

Grano línea : 9,800 × 28   |   Grano pedido: 4,922 × 16


---
# Fase 4 · Definición de KPIs de negocio

## 8. Diccionario de KPIs de negocio

Los diez KPIs cubren las cinco preguntas clave y los cuatro ángulos del negocio:
**crecimiento**, **valor de cliente**, **territorio**, **catálogo**, **estacionalidad**
y **operación**. Cada uno declara fórmula, unidad, meta, tendencia observada,
hipótesis que valida y justificación de negocio

In [4]:
# ---------------------------------------------------------------------------
# 8.1 Cálculo de los KPIs sobre los datos limpios
# ---------------------------------------------------------------------------
ventas_anio = df.groupby("Anio")["Sales"].sum()
pedidos_anio = pedidos.groupby("Anio").size()
ticket_anio = ventas_anio / pedidos_anio
clientes_anio = pedidos.groupby("Anio")["Cliente"].nunique()
lineas_anio = df.groupby("Anio").size()

def cagr(inicial, final, periodos):
    return (final / inicial) ** (1 / periodos) - 1

ventas_cliente = pedidos.groupby("Cliente")["Ventas"].sum()
pedidos_cliente = pedidos.groupby("Cliente").size()
cuota_region = df.groupby("Region")["Sales"].sum() / df["Sales"].sum()
cuota_estado = df.groupby("State")["Sales"].sum() / df["Sales"].sum()
cuota_subcat = df.groupby("Sub-Category")["Sales"].sum() / df["Sales"].sum()

kpis = {
    "K1 Ingresos totales": df["Sales"].sum(),
    "K2 Pedidos": pedidos.shape[0],
    "K3 Ticket medio": pedidos["Ventas"].mean(),
    "K4 Crecimiento anual de ingresos (CAGR 2016-2018)": cagr(ventas_anio[2016], ventas_anio[2018], 2),
    "K5 Cuota de la región líder (West)": cuota_region.max(),
    "K6 Índice de concentración (HHI) por estado": float((cuota_estado ** 2).sum()),
    "K7 Tasa de recompra": float((pedidos_cliente > 1).mean()),
    "K8 Índice de estacionalidad Q4": float(
        df[df.EsQ4]["Sales"].sum() / 92 / (df[~df.EsQ4]["Sales"].sum() / 273)),
    "K9 Cumplimiento logístico (≤ 4 días)": float((df["DiasEnvio"] <= 4).mean()),
    "K10 Ticket del segmento Home Office": float(pedidos.loc[pedidos.Segmento == "Home Office", "Ventas"].mean()),
}
for k, v in kpis.items():
    formato = f"{v:,.4f}" if v < 1 else f"{v:,.2f}"
    print(f"{k:<52} {formato}")

K1 Ingresos totales                                  2,261,536.78
K2 Pedidos                                           4,922.00
K3 Ticket medio                                      459.48
K4 Crecimiento anual de ingresos (CAGR 2016-2018)    0.2536
K5 Cuota de la región líder (West)                   0.3140
K6 Índice de concentración (HHI) por estado          0.0791
K7 Tasa de recompra                                  0.9836
K8 Índice de estacionalidad Q4                       1.86
K9 Cumplimiento logístico (≤ 4 días)                 0.5988
K10 Ticket del segmento Home Office                  475.37


In [5]:
# ---------------------------------------------------------------------------
# 8.2 Diccionario formal de KPIs
# ---------------------------------------------------------------------------
diccionario = pd.DataFrame([
    dict(KPI="K1 · Ingresos totales",
         Formula="SUM(Sales)",
         Unidad="USD", Meta="Crecimiento ≥ 25 % anual",
         Observado=f"{df['Sales'].sum():,.0f} en el periodo",
         Hipotesis="H1", Pregunta="P1",
         Justificacion="Mide el tamaño del negocio; es la base de todo cociente posterior."),
    dict(KPI="K2 · Pedidos",
         Formula="COUNTROWS(DISTINCT Order ID)",
         Unidad="Pedidos", Meta="+60 % acumulado 2016-2018",
         Observado=f"{pedidos.shape[0]:,} totales; +63,0 % 2018 vs 2016",
         Hipotesis="H1", Pregunta="P1",
         Justificacion="Aísla el componente volumen del crecimiento del ingreso."),
    dict(KPI="K3 · Ticket medio",
         Formula="SUM(Sales) / COUNTROWS(DISTINCT Order ID)",
         Unidad="USD", Meta="≥ 450 USD",
         Observado=f"{pedidos['Ventas'].mean():,.2f} global; 434,71 en 2018",
         Hipotesis="H1", Pregunta="P1",
         Justificacion="Es la palanca de valor por transacción; su estancamiento redirige "
                       "el esfuerzo hacia frecuencia y adquisición."),
    dict(KPI="K4 · Crecimiento anual de ingresos (CAGR)",
         Formula="(Ventas 2018 / Ventas 2016)^(1/2) - 1",
         Unidad="%", Meta="≥ 20 %",
         Observado=f"{cagr(ventas_anio[2016], ventas_anio[2018], 2)*100:,.1f} %",
         Hipotesis="H1", Pregunta="P1",
         Justificacion="Normaliza el crecimiento por el efecto de la base parcial de 2015."),
    dict(KPI="K5 · Cuota de la región líder",
         Formula="Ventas región / Ventas totales",
         Unidad="%", Meta="Mantener ≥ 30 % sin perder crecimiento",
         Observado=f"West {cuota_region.max()*100:,.1f} %",
         Hipotesis="H2", Pregunta="P2",
         Justificacion="Detecta dependencia excesiva de un territorio."),
    dict(KPI="K6 · Índice de concentración (HHI) por estado",
         Formula="Σ (cuota estado)²",
         Unidad="Índice 0-1", Meta="< 0,10 (mercado diversificado)",
         Observado=f"{(cuota_estado**2).sum():,.4f}",
         Hipotesis="H3", Pregunta="P3",
         Justificacion="Cuantifica el riesgo geográfico de la facturación."),
    dict(KPI="K7 · Tasa de recompra",
         Formula="Clientes con > 1 pedido / Clientes totales",
         Unidad="%", Meta="≥ 90 %",
         Observado=f"{(pedidos_cliente > 1).mean()*100:,.1f} %",
         Hipotesis="H1", Pregunta="P1",
         Justificacion="Un padrón que recompra sostiene el volumen; su caída anticipa "
                       "una contracción del ingreso."),
    dict(KPI="K8 · Índice de estacionalidad Q4",
         Formula="(Venta diaria oct-dic) / (Venta diaria ene-sep)",
         Unidad="Veces", Meta="Planificar capacidad ≥ 1,8×",
         Observado=f"{(df[df.EsQ4]['Sales'].sum()/92)/(df[~df.EsQ4]['Sales'].sum()/273):,.2f}×",
         Hipotesis="H4", Pregunta="P4",
         Justificacion="Es el input directo para dimensionar inventario, personal y transporte."),
    dict(KPI="K9 · Cumplimiento logístico (≤ 4 días)",
         Formula="Líneas con DiasEnvio ≤ 4 / Total de líneas",
         Unidad="%", Meta="≥ 80 %",
         Observado=f"{(df['DiasEnvio'] <= 4).mean()*100:,.1f} %",
         Hipotesis="H5", Pregunta="P5",
         Justificacion="El nivel de servicio sostiene la recompra; se mide sin costo de envío "
                       "disponible en el dataset."),
    dict(KPI="K10 · Ticket del segmento de mayor valor",
         Formula="SUM(Sales) / Pedidos del segmento",
         Unidad="USD", Meta="Ampliar el mix de Home Office",
         Observado=f"{pedidos.loc[pedidos.Segmento=='Home Office','Ventas'].mean():,.2f} (Home Office)",
         Hipotesis="H1", Pregunta="P1",
         Justificacion="Home Office muestra el ticket más alto pese a ser el segmento más pequeño: "
                       "señal de oportunidad de mezcla."),
])
diccionario

,KPI,Formula,Unidad,Meta,Observado,Hipotesis,Pregunta,Justificacion
0,K1 · Ingresos totales,SUM(Sales),USD,Crecimiento ≥ 25 % anual,"2,261,537 en el periodo",H1,P1,Mide el tamaño del negocio; es la base de todo...
1,K2 · Pedidos,COUNTROWS(DISTINCT Order ID),Pedidos,+60 % acumulado 2016-2018,"4,922 totales; +63,0 % 2018 vs 2016",H1,P1,Aísla el componente volumen del crecimiento de...
2,K3 · Ticket medio,SUM(Sales) / COUNTROWS(DISTINCT Order ID),USD,≥ 450 USD,"459.48 global; 434,71 en 2018",H1,P1,Es la palanca de valor por transacción; su est...
3,K4 · Crecimiento anual de ingresos (CAGR),(Ventas 2018 / Ventas 2016)^(1/2) - 1,%,≥ 20 %,25.4 %,H1,P1,Normaliza el crecimiento por el efecto de la b...
4,K5 · Cuota de la región líder,Ventas región / Ventas totales,%,Mantener ≥ 30 % sin perder crecimiento,West 31.4 %,H2,P2,Detecta dependencia excesiva de un territorio.
5,K6 · Índice de concentración (HHI) por estado,Σ (cuota estado)²,Índice 0-1,"< 0,10 (mercado diversificado)",0.0791,H3,P3,Cuantifica el riesgo geográfico de la facturac...
6,K7 · Tasa de recompra,Clientes con > 1 pedido / Clientes totales,%,≥ 90 %,98.4 %,H1,P1,Un padrón que recompra sostiene el volumen; su...
7,K8 · Índice de estacionalidad Q4,(Venta diaria oct-dic) / (Venta diaria ene-sep),Veces,"Planificar capacidad ≥ 1,8×",1.86×,H4,P4,Es el input directo para dimensionar inventari...
8,K9 · Cumplimiento logístico (≤ 4 días),Líneas con DiasEnvio ≤ 4 / Total de líneas,%,≥ 80 %,59.9 %,H5,P5,El nivel de servicio sostiene la recompra; se ...
9,K10 · Ticket del segmento de mayor valor,SUM(Sales) / Pedidos del segmento,USD,Ampliar el mix de Home Office,475.37 (Home Office),H1,P1,Home Office muestra el ticket más alto pese a ...


In [6]:
# ---------------------------------------------------------------------------
# 8.3 Evidencia cuantitativa de cada KPI: series de soporte
# ---------------------------------------------------------------------------
evolucion = pd.DataFrame({
    "Año": ventas_anio.index,
    "Ingresos (K1)": ventas_anio.values,
    "Pedidos (K2)": pedidos_anio.values,
    "Ticket medio (K3)": ticket_anio.values,
    "Clientes activos": clientes_anio.values,
    "Líneas por pedido": (lineas_anio / pedidos_anio).values,
})
evolucion["Var. ingresos"] = evolucion["Ingresos (K1)"].pct_change().map(
    lambda v: "—" if pd.isna(v) else f"{v*100:+.1f} %")
evolucion["Var. pedidos"] = evolucion["Pedidos (K2)"].pct_change().map(
    lambda v: "—" if pd.isna(v) else f"{v*100:+.1f} %")
evolucion["Var. ticket"] = evolucion["Ticket medio (K3)"].pct_change().map(
    lambda v: "—" if pd.isna(v) else f"{v*100:+.1f} %")

print("Nota: 2015 es un año incompleto del dataset y su ticket medio es atípicamente alto; "
      "las comparaciones de crecimiento se anclan en 2016.")
display(evolucion.round(2))

Nota: 2015 es un año incompleto del dataset y su ticket medio es atípicamente alto; las comparaciones de crecimiento se anclan en 2016.


,Año,Ingresos (K1),Pedidos (K2),Ticket medio (K3),Clientes activos,Líneas por pedido,Var. ingresos,Var. pedidos,Var. ticket
0,2015,"479,856.2100",947,506.7100,589,2.0600,—,—,—
1,2016,"459,436.0100",1019,450.8700,567,2.0200,-4.3 %,+7.6 %,-11.0 %
2,2017,"600,192.5500",1295,463.4700,635,1.9600,+30.6 %,+27.1 %,+2.8 %
3,2018,"722,052.0200",1661,434.7100,690,1.9600,+20.3 %,+28.3 %,-6.2 %
